# Crowd Cooling Review Notebook

This notebook is for visual review of the current evaluation artifacts.

- Bottom camera: count + hotspot ROI + mist decision
- Side camera: forward crowd gating
- Dual camera: fused smoke-run logs


In [ ]:
from __future__ import annotations

import ast
import json
from functools import lru_cache
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display
from PIL import Image, ImageDraw
from ultralytics import YOLO

CWD = Path.cwd()
ROOT = CWD if (CWD / 'eval').exists() else CWD.parent
RUN_CANDIDATES = [
    ROOT / 'eval/runs/smoke_yolov8n_val_v2',
    ROOT / 'eval/runs/smoke_yolov8n_val',
]
RUN_DIR = next((path for path in RUN_CANDIDATES if path.exists()), None)
if RUN_DIR is None:
    raise FileNotFoundError('No evaluation run found under eval/runs/.')

SUMMARY_PATH = RUN_DIR / 'summary.json'
PER_FRAME_PATH = RUN_DIR / 'per_frame.csv'
SUMMARY = json.loads(SUMMARY_PATH.read_text(encoding='utf-8'))
DF = pd.read_csv(PER_FRAME_PATH)

display(Markdown(f'**Workspace Root:** `{ROOT}`'))
display(Markdown(f'**Evaluation Run:** `{RUN_DIR}`'))
display(pd.DataFrame({'available_eval_runs': [p.name for p in sorted((ROOT / 'eval/runs').glob('*'))]}))


In [ ]:
count_metrics = SUMMARY['count_metrics']
latency = SUMMARY['latency_metrics']
overview = pd.DataFrame([
    {
        'model': SUMMARY['model_path'],
        'split': SUMMARY['split'],
        'frames': SUMMARY['num_frames'],
        'accuracy_count': count_metrics['accuracy_count'],
        'mae': count_metrics['mae'],
        'rmse': count_metrics['rmse'],
        'total_ms_mean': latency['total_ms']['mean'],
        'total_ms_p90': latency['total_ms']['p90'],
    }
])
display(overview)

review_columns = ['image_path', 'gt_count', 'pred_count', 'abs_error', 'rel_error', 'pass_20']
display(DF[review_columns].head(12))


In [ ]:
def label_path_from_image(image_path: Path) -> Path:
    parts = list(image_path.parts)
    idx = parts.index('images')
    parts[idx] = 'labels'
    return Path(*parts).with_suffix('.txt')


def load_gt_boxes(image_path: Path):
    label_path = label_path_from_image(image_path)
    if not label_path.exists():
        return []
    with Image.open(image_path) as img:
        width, height = img.size
    boxes = []
    for line in label_path.read_text(encoding='utf-8').splitlines():
        if not line.strip():
            continue
        _, xc, yc, bw, bh = map(float, line.split())
        box_w = bw * width
        box_h = bh * height
        x1 = (xc * width) - (box_w / 2.0)
        y1 = (yc * height) - (box_h / 2.0)
        x2 = x1 + box_w
        y2 = y1 + box_h
        boxes.append((x1, y1, x2, y2))
    return boxes


@lru_cache(maxsize=2)
def get_model(model_path: str):
    return YOLO(model_path)


def render_row(row, conf=None, iou=None, imgsz=None):
    image_path = Path(row['image_path'])
    image = Image.open(image_path).convert('RGB')
    draw = ImageDraw.Draw(image)
    gt_boxes = load_gt_boxes(image_path)
    for box in gt_boxes:
        draw.rectangle(box, outline=(255, 165, 0), width=2)

    model = get_model(SUMMARY['model_path'])
    result = model.predict(
        str(image_path),
        conf=SUMMARY['conf'] if conf is None else conf,
        iou=SUMMARY['iou'] if iou is None else iou,
        imgsz=SUMMARY['imgsz'] if imgsz is None else imgsz,
        device=SUMMARY['device'],
        verbose=False,
    )[0]
    if result.boxes is not None:
        for xyxy in result.boxes.xyxy.cpu().numpy():
            draw.rectangle([float(v) for v in xyxy], outline=(0, 255, 0), width=3)

    roi_u = float(row['roi_u'])
    roi_v = float(row['roi_v'])
    draw.line([(roi_u - 12, roi_v), (roi_u + 12, roi_v)], fill=(255, 0, 0), width=3)
    draw.line([(roi_u, roi_v - 12), (roi_u, roi_v + 12)], fill=(255, 0, 0), width=3)

    overlay = [
        f"GT={int(row['gt_count'])} Pred={int(row['pred_count'])} Pass={int(row['pass_20'])}",
        f"RelErr={float(row['rel_error']):.2f} Density={float(row['density_score']):.2f}",
        f"Role={row.get('camera_role', 'bottom')} Reason={row.get('decision_reason', '')}",
    ]
    y = 10
    for text in overlay:
        draw.rectangle([8, y, min(image.size[0] - 8, 980), y + 24], fill=(0, 0, 0))
        draw.text((14, y + 4), text, fill=(255, 255, 255))
        y += 28
    return image


In [ ]:
worst = DF.sort_values(['pass_20', 'abs_error'], ascending=[True, False]).head(6)
display(worst[['image_path', 'gt_count', 'pred_count', 'abs_error', 'rel_error', 'pass_20']])

fig, axes = plt.subplots(len(worst), 1, figsize=(16, 5 * len(worst)))
if len(worst) == 1:
    axes = [axes]
for ax, (_, row) in zip(axes, worst.iterrows()):
    ax.imshow(render_row(row))
    ax.set_title(Path(row['image_path']).name)
    ax.axis('off')
plt.tight_layout()


In [ ]:
passes = DF[DF['pass_20'] == 1].copy()
if passes.empty:
    display(Markdown('**No pass frames exist in the current smoke run.**'))
else:
    display(passes[['image_path', 'gt_count', 'pred_count', 'abs_error', 'rel_error']].head())
    sample = passes.head(3)
    fig, axes = plt.subplots(len(sample), 1, figsize=(16, 5 * len(sample)))
    if len(sample) == 1:
        axes = [axes]
    for ax, (_, row) in zip(axes, sample.iterrows()):
        ax.imshow(render_row(row))
        ax.set_title(Path(row['image_path']).name)
        ax.axis('off')
    plt.tight_layout()


In [ ]:
side_log = ROOT / 'deploy/runs/smoke_side/decision_log.csv'
dual_log = ROOT / 'deploy/runs/smoke_dual/dual_camera_log.csv'

if side_log.exists():
    display(Markdown('## Side Camera Smoke Log'))
    display(pd.read_csv(side_log).head(10))

if dual_log.exists():
    display(Markdown('## Dual Camera Smoke Log'))
    display(pd.read_csv(dual_log).head(10))


## Optional next runs

Use the shell commands below in a notebook cell if you want bigger artifacts than the current smoke run.

```python
!python eval/evaluate_model.py --model MovingDroneCrowd/yolov8n.pt --data datasets/processed/mdc_head_yolo/mdc_head_yolo.yaml --split val --conf 0.10 --iou 0.70 --imgsz 640 --device 0 --output-dir eval/runs/manual_review --max-images 50
!python eval/render_examples.py --per-frame-csv eval/runs/manual_review/per_frame.csv --model MovingDroneCrowd/yolov8n.pt --output-dir docs/figures/manual_review --conf 0.10 --iou 0.70 --imgsz 640 --device 0
```
